## CellLineSelector — Data Harmonisation Pipeline


In [ ]:
"""
CellLineSelector — Data Harmonisation Pipeline
================================================
Multi-omics cancer cell line identity harmonisation and feature matrix
assembly, integrating DepMap/CCLE, Cellosaurus, GEO, and HPA sources into
a unified, model_id-indexed structure for downstream GAT / MOFA+ modelling.

Author: Tee
Project: CellLineSelector (AstraZeneca collaboration, Univ. of Birmingham/Bristol, 2025)

Pipeline stages:
    1. Load cleaned parquet tables
    2. Identity harmonisation (RRID / CVCL / ACH / profile / GEO linking)
    3. Coverage auditing across modalities
    4. Per-modality feature matrix assembly (model_id-indexed)
    5. AnnData / MuData construction
"""

import sys, os
sys.path.insert(0, r"C:\Disertation\UoB-GeneTraceAI-25-26\src\scripts")


import re
import pandas as pd
from data_utils import load_clean_parquets, preview

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

### Load cleaned parquet tables

In [ ]:
# ---------------------------------------------------------------------------
# Stage 1: Load cleaned parquet tables
# ---------------------------------------------------------------------------
"""
Loads all 14 pre-cleaned source tables via load_clean_parquets(), which reads
from the cleaned_track_data directory. Each table has already passed through
lower_all() and normalize_cellname() in prior cleaning notebooks — columns
are lowercased, string values are lowercased, and cell line names have
separators stripped.

Tables loaded:
    hpa_rna         - Human Protein Atlas RNA expression (wide, model-level)
    depmap_expr     - DepMap expression matrix (profile-indexed)
    geo_expr        - GEO expression matrix (GSM sample-indexed)
    proteomics      - CCLE proteomics (Nusinow et al. 2020, log-ratio)
    protein_map     - Protein/gene ID mapping reference
    fusions         - Gene fusion event table (model_id-keyed)
    mutations       - Mutation event table (profile_id-keyed)
    cellosaurus     - Cellosaurus identity reference (accession <-> name)
    depmap_profiles - DepMap profile registry (model_id <-> profile_id)
    sample_info     - DepMap primary sample/ACH roster (authoritative)
    geo_info        - GEO sample metadata (cellosaurus_id, geo_accession, link_status)
    hpa_desc        - HPA description/metadata table
    metabolomics    - CCLE metabolomics
    mirna           - miRNA expression (wide, CCLE {name}_{tissue} headers)
    signatures      - Cell line signature/score table
"""

tables = load_clean_parquets()

hpa_rna         = tables["hpa_rna"]
depmap_expr     = tables["depmap_expr"]
geo_expr        = tables["geo_expr"]
proteomics      = tables["proteomics"]
protein_map     = tables["protein_map"]
fusions         = tables["fusions"]
mutations       = tables["mutations"]
cellosaurus     = tables["cellosaurus"]
depmap_profiles = tables["depmap_profiles"]
sample_info     = tables["sample_info"]
geo_info        = tables["geo_info"]
hpa_desc        = tables["hpa_desc"]
metabolomics    = tables["metabolomics"]
mirna           = tables["mirna"]
signatures      = tables["signatures"]

In [ ]:
def lower_all(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].astype("string").str.strip().str.lower()
    return df

In [ ]:
# ---------------------------------------------------------------------------
# Stage 2: Identity harmonisation — cellosaurus cleaning
# ---------------------------------------------------------------------------
"""
Normalises the cellosaurus table: lowercases all string data and column
headers, then strips any prefix (e.g. 'cvcl:') or suffix (e.g. version
tags, whitespace) from cellosaurus_accession so every accession is left
in bare 'cvcl_xxxx' form — matching the join-key format used across
sample_info, geo_info, and the rrid roster.
"""

cellosaurus = lower_all(cellosaurus)

cellosaurus["cellosaurus_accession"] = (
    cellosaurus["cellosaurus_accession"]
    .astype("string")
    .str.strip()
    .str.replace(r"^cvcl[:_\-\s]*", "cvcl_", regex=True)   # normalise any prefix to 'cvcl_'
    .str.replace(r"[^a-z0-9_]+$", "", regex=True)           # strip trailing junk/suffix
)

print("rows:", len(cellosaurus))
print("sample accessions:", cellosaurus["cellosaurus_accession"].dropna().head(5).tolist())
print("missing accession:", cellosaurus["cellosaurus_accession"].isna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 3: Identity harmonisation — sample_info rrid cleaning
# ---------------------------------------------------------------------------
"""
Normalises sample_info's rrid column the same way as cellosaurus_accession:
lowercases everything, strips any prefix (e.g. 'rrid:') or suffix, and
leaves a bare 'cvcl_xxxx' value so it joins cleanly against
cellosaurus_accession on the same footing.
"""

sample_info = lower_all(sample_info)

sample_info["rrid"] = (
    sample_info["rrid"]
    .astype("string")
    .str.strip()
    .str.replace(r"^rrid[:_\-\s]*", "", regex=True)          # drop 'rrid:' prefix
    .str.replace(r"^cvcl[:_\-\s]*", "cvcl_", regex=True)      # normalise to 'cvcl_' prefix
    .str.replace(r"[^a-z0-9_]+$", "", regex=True)             # strip trailing junk/suffix
)

print("rows:", len(sample_info))
print("sample rrid:", sample_info["rrid"].dropna().head(5).tolist())
print("missing rrid:", sample_info["rrid"].isna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 4: Identity harmonisation — fill missing rrid via cellosaurus name match
# ---------------------------------------------------------------------------
"""
Fills missing rrid values in sample_info using cellosaurus as a name-based
backstop. Builds a cell_line_name -> CVCL lookup directly from cellosaurus,
keeping only unambiguous names (exactly one CVCL per name) to avoid
arbitrarily picking among short-code collisions. Only rows with a missing
rrid are touched; existing authoritative rrid values are never overwritten.
Before/after counts audit exactly how many rows were recovered.
"""

# name -> single unambiguous CVCL from cellosaurus
acc_by_name = (cellosaurus.dropna(subset=["cellosaurus_accession"])
                 .groupby("cellosaurus_cell_line_name")["cellosaurus_accession"]
                 .agg(lambda s: set(s.dropna())))
unambiguous = acc_by_name[acc_by_name.map(len) == 1].map(lambda s: next(iter(s)))

# fill only where rrid is missing
mask = sample_info["rrid"].isna()
fill = sample_info.loc[mask, "stripped_cell_line_name"].map(unambiguous)

before_missing = mask.sum()
sample_info.loc[mask, "rrid"] = fill

after_missing = sample_info["rrid"].isna().sum()
print("was missing:   ", before_missing)
print("newly filled:  ", before_missing - after_missing)
print("still missing: ", after_missing)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 5: Identity harmonisation — aggregate rrid per ACH (depmap_id)
# ---------------------------------------------------------------------------
"""
Aggregates sample_info over depmap_id (ACH) to check the rrid -> ACH
relationship. Since sample_info is expected to be effectively one row per
ACH, n_rrid should be 1 for nearly every model; any ACH with n_rrid > 1
flags a genuine identity ambiguity (mirroring the known u-251 mg case),
and n_rrid == 0 flags an ACH with no resolved CVCL after the fill step.
"""

rrid_by_ach = (sample_info.dropna(subset=["depmap_id"])
                 .groupby("depmap_id")["rrid"]
                 .agg(n_rrid=lambda s: s.dropna().nunique(),
                      rrids=lambda s: sorted(set(s.dropna())))
                 .reset_index()
                 .sort_values("n_rrid", ascending=False))

print("distinct ach ids:", len(rrid_by_ach))
print("ach with >1 rrid: ", (rrid_by_ach["n_rrid"] > 1).sum())
print("ach with 0 rrid:  ", (rrid_by_ach["n_rrid"] == 0).sum())
rrid_by_ach

In [ ]:
# ---------------------------------------------------------------------------
# Stage 6: Identity harmonisation — check modelid <-> profileid cardinality
# ---------------------------------------------------------------------------
"""
Checks the cardinality of the modelid <-> profileid relationship in
depmap_profiles in both directions:
    - profile -> model_id relationship (is each profile tied to one model?)
    - model_id -> profile relationship (does each model have one profile?)
This determines whether profile_ids can be treated as a single value per
ACH, or must be kept as a set.
"""

profile_to_model = depmap_profiles.dropna(subset=["profileid"]).groupby("profileid")["modelid"].nunique()
model_to_profile = depmap_profiles.dropna(subset=["modelid"]).groupby("modelid")["profileid"].nunique()

print("=== profile -> model_id relationship ===")
print("profiles with >1 model_id:", (profile_to_model > 1).sum(), "/", len(profile_to_model))
print("max model_ids per profile:", profile_to_model.max())
print()
print("=== model_id -> profile relationship ===")
print("models with >1 profile_id:", (model_to_profile > 1).sum(), "/", len(model_to_profile))
print("max profile_ids per model:", model_to_profile.max())
print()
is_one_to_one = (profile_to_model.max() == 1) and (model_to_profile.max() == 1)
print("strictly 1:1 relationship:", is_one_to_one)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 7: Identity harmonisation — attach profile_id(s) per ACH
# ---------------------------------------------------------------------------
"""
Aggregates depmap_profiles over modelid to collect the set of profile_ids
per ACH (a model can legitimately have multiple profiles — RNA, WES, WGS —
so n_profile_id > 1 is expected, not a flag). Left-joins onto rrid_by_ach
so every ACH in the roster is preserved, with unmatched ACHs surfaced as
NaN rather than silently dropped.
"""

prof_by_ach = (depmap_profiles.dropna(subset=["modelid"])
                 .groupby("modelid")["profileid"]
                 .agg(n_profile_id="nunique",
                      profile_ids=lambda s: sorted(set(s.dropna())))
                 .reset_index()
                 .rename(columns={"modelid": "depmap_id"}))   # modelid, not model_id

rrid_by_ach = rrid_by_ach.merge(prof_by_ach, on="depmap_id", how="left")

print("ach in rrid_by_ach:        ", len(rrid_by_ach))
print("ach with profile_id(s):    ", rrid_by_ach["n_profile_id"].notna().sum())
print("ach with >1 profile_id:    ", (rrid_by_ach["n_profile_id"] > 1).sum())
print("ach with no profile match: ", rrid_by_ach["n_profile_id"].isna().sum())
rrid_by_ach

In [ ]:
# ---------------------------------------------------------------------------
# Stage 8: Identity harmonisation — check cardinality in geo_info
# ---------------------------------------------------------------------------
"""
Checks the cardinality of the geo_accession <-> cellosaurus_id relationship
in geo_info in both directions:
    - geo_accession -> cellosaurus_id (does each GEO sample resolve to one CVCL?)
    - cellosaurus_id -> geo_accession (how many GEO samples per CVCL/line?)
geo_info is sample-grain, so cellosaurus_id -> geo_accession is expected to
be genuinely one-to-many (a line profiled across multiple GEO samples);
geo_accession -> cellosaurus_id should be 1:1 since one sample belongs to
one line.
"""

geo_to_cvcl = geo_info.dropna(subset=["geo_accession"]).groupby("geo_accession")["cellosaurus_id"].nunique()
cvcl_to_geo = geo_info.dropna(subset=["cellosaurus_id"]).groupby("cellosaurus_id")["geo_accession"].nunique()

print("=== geo_accession -> cellosaurus_id relationship ===")
print("geo_accessions with >1 cellosaurus_id:", (geo_to_cvcl > 1).sum(), "/", len(geo_to_cvcl))
print("max cellosaurus_id per geo_accession:  ", geo_to_cvcl.max())
print()
print("=== cellosaurus_id -> geo_accession relationship ===")
print("cellosaurus_ids with >1 geo_accession:", (cvcl_to_geo > 1).sum(), "/", len(cvcl_to_geo))
print("max geo_accession per cellosaurus_id: ", cvcl_to_geo.max())
print()
is_one_to_one = (geo_to_cvcl.max() == 1) and (cvcl_to_geo.max() == 1)
print("strictly 1:1 relationship:", is_one_to_one)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 9: Identity harmonisation — attach geo_accession(s) per ACH
# ---------------------------------------------------------------------------
"""
Builds a cvcl -> geo_accessions lookup from geo_info (one CVCL/line can
legitimately span many GEO samples/studies). Explodes rrid_by_ach's rrids
set so each (ACH, single CVCL) pair is its own row, maps each CVCL to its
GEO accessions, then re-collapses back to one row per ACH by unioning the
accessions across all of that ACH's CVCLs. Left-joins onto rrid_by_ach so
every ACH is preserved, including those with no GEO match.
"""

# lookup: cvcl -> geo_accessions list
geo_by_cvcl = (geo_info.dropna(subset=["cellosaurus_id"])
                 .groupby("cellosaurus_id")["geo_accession"]
                 .agg(n_geo="nunique",
                      geo_accessions=lambda s: sorted(set(s.dropna())))
                 .reset_index()
                 .sort_values("n_geo", ascending=False))

print("distinct cvcl:", len(geo_by_cvcl))
print("cvcl with >1 geo_accession:", (geo_by_cvcl["n_geo"] > 1).sum())
print("cvcl with exactly 1:       ", (geo_by_cvcl["n_geo"] == 1).sum())
geo_by_cvcl.head(20)

geo_lookup = geo_by_cvcl.set_index("cellosaurus_id")["geo_accessions"]

# explode rrids so each (ach, single-cvcl) pair is its own row, map, then re-collapse
exploded = (rrid_by_ach[["depmap_id", "rrids"]]
              .explode("rrids")
              .rename(columns={"rrids": "cvcl"}))
exploded["geo_accessions"] = exploded["cvcl"].map(geo_lookup)

# collapse back to one row per ach: union all geo_accessions across its cvcls
def _union(lists):
    out = set()
    for x in lists.dropna():
        out.update(x)
    return sorted(out)

geo_per_ach = (exploded.groupby("depmap_id")["geo_accessions"]
                 .agg(geo_accessions=_union)
                 .reset_index())
geo_per_ach["n_geo"] = geo_per_ach["geo_accessions"].map(len)

# attach back to rrid_by_ach (left join keeps unmatched ach visible)
rrid_by_ach = rrid_by_ach.merge(geo_per_ach, on="depmap_id", how="left")

# audit
print("ach total:              ", len(rrid_by_ach))
rrid_by_ach.head(20)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 10: Coverage auditing — mutations profileid coverage
# ---------------------------------------------------------------------------
"""
Aggregates mutations over profileid to check row density per profile
(mutations is an event table, so many rows per profile is expected), then
checks whether every profileid in mutations is present in the rrid_by_ach
roster's profile_ids. Any missing profileids are orphaned mutation records
that don't trace back to a known ACH via the roster, and are flagged
rather than silently dropped.
"""

mut_by_profile = (mutations.groupby("profileid")
                    .size()
                    .reset_index(name="n_rows")
                    .sort_values("n_rows", ascending=False))

print("distinct profileid:", len(mut_by_profile))
print("total mutation rows:", len(mutations))
mut_by_profile.head(20)

# flatten all profile_ids from rrid_by_ach into one set
roster_profiles = set(rrid_by_ach["profile_ids"].explode().dropna())

mut_profiles = set(mutations["profileid"].dropna())

missing = mut_profiles - roster_profiles
present = mut_profiles & roster_profiles

print("distinct mutation profileids:", len(mut_profiles))
print("present in rrid_by_ach:      ", len(present))
print("missing from rrid_by_ach:    ", len(missing))
print("all present:", mut_profiles.issubset(roster_profiles))

if missing:
    print("sample missing:", list(missing)[:20])

In [ ]:
# ---------------------------------------------------------------------------
# Stage 11: Coverage auditing — fusions modelid coverage
# ---------------------------------------------------------------------------
"""
Checks whether every modelid (ACH) in fusions is present in the rrid_by_ach
roster. Unlike mutations, fusions is keyed directly on modelid, so this
compares straight against depmap_id with no profile->ACH hop needed. Any
missing ACHs are fusion records that don't trace back to a known model in
the roster, flagged rather than silently dropped.
"""

roster_achs = set(rrid_by_ach["depmap_id"].dropna())
fusion_achs = set(fusions["modelid"].dropna())

missing = fusion_achs - roster_achs
present = fusion_achs & roster_achs

print("distinct fusion modelids:", len(fusion_achs))
print("present in rrid_by_ach:  ", len(present))
print("missing:                 ", len(missing))
print("all present:", fusion_achs.issubset(roster_achs))

if missing:
    print("sample missing:", list(missing)[:20])

In [ ]:
# ---------------------------------------------------------------------------
# Stage 12: Coverage auditing — check table orientation (depmap_expr, geo_expr)
# ---------------------------------------------------------------------------
"""
Checks the current orientation of depmap_expr and geo_expr — confirming
whether cell lines / samples are rows and genes are columns, or the
reverse. This determines whether a transpose is needed before these
tables can be assembled into the model_id-indexed feature matrices.
"""

for name, df in [("depmap_expr", depmap_expr), ("geo_expr", geo_expr)]:
    print(f"=== {name} ===")
    print("shape:", df.shape)
    print("index name:  ", df.index.name)
    print("index sample:", df.index[:5].tolist())
    print("columns sample:", df.columns[:5].tolist())
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Stage 13: Feature matrix assembly — reshape geo_expr to sample-rows/gene-columns
# ---------------------------------------------------------------------------
"""
Transposes geo_expr from gene-rows/sample-columns into the target wide
form: one row per GSM sample, one column per gene. 'gene' is set as the
index before transposing so it becomes the column headers post-transpose,
and the former column headers (GSM sample IDs) become the row index.
similar format as depmap expression.
"""

geo_wide = geo_expr.set_index("gene").T
geo_wide.index.name = "sample"     # rows are GSM samples, not yet collapsed to cell_line
geo_wide.columns.name = None

print(geo_wide.shape)
geo_wide.head()

In [ ]:
# ---------------------------------------------------------------------------
# Stage 14: Feature matrix assembly — attach model_id to geo_expr
# ---------------------------------------------------------------------------
"""
Adds model_id to geo_wide (geo_expr, GSM-sample-indexed) by mapping each
GSM sample to its ACH via rrid_by_ach's geo_accessions set. Since one ACH
can span multiple GEO accessions, the roster is exploded to a flat
gsm -> depmap_id lookup before mapping. A small number of GSMs map to more
than one ACH (ambiguous CVCL->ACH cases) — these are preserved as a
model_id_set column rather than silently collapsed by drop_duplicates, so
the ambiguity stays visible rather than being resolved arbitrarily.
"""

exploded_gsm = (rrid_by_ach[["depmap_id", "geo_accessions"]]
                  .explode("geo_accessions")
                  .dropna(subset=["geo_accessions"]))

# full set of model_ids per gsm (preserves ambiguity)
gsm_to_model_set = (exploded_gsm.groupby("geo_accessions")["depmap_id"]
                       .agg(lambda s: sorted(set(s))))

# single model_id per gsm, only unambiguous ones (n==1)
gsm_to_model_single = gsm_to_model_set[gsm_to_model_set.map(len) == 1].map(lambda s: s[0])

geo_wide = geo_wide.reset_index() if "sample" not in geo_wide.columns else geo_wide

geo_wide["model_id"] = geo_wide["sample"].map(gsm_to_model_set)

print("rows:", len(geo_wide))
print("unmatched:                     ", geo_wide["model_id"].isna().sum())
geo_expr = geo_wide
geo_expr

In [ ]:
# ---------------------------------------------------------------------------
# Stage 15: Feature matrix assembly — attach model_id to depmap_expr
# ---------------------------------------------------------------------------
"""
Adds model_id to depmap_expr (profile-indexed) by mapping each profileid
to its ACH via rrid_by_ach's profile_ids set. Since a model_id -> profile
relationship can be one-to-many (RNA, WES, WGS profiles) but profile ->
model_id should be 1:1 per the earlier cardinality check, this mirrors the
geo_expr pattern: full model_id_set preserved for visibility, plus a clean
single model_id column where unambiguous.
"""

depmap_expr = depmap_expr.reset_index() if "profileid" not in depmap_expr.columns else depmap_expr
if depmap_expr.columns[0] != "profileid" and "profileid" not in depmap_expr.columns:
    depmap_expr = depmap_expr.rename(columns={depmap_expr.columns[0]: "profileid"})

exploded_profile = (rrid_by_ach[["depmap_id", "profile_ids"]]
                       .explode("profile_ids")
                       .dropna(subset=["profile_ids"]))

# full set of model_ids per profileid (preserves ambiguity if any)
profile_to_model_set = (exploded_profile.groupby("profile_ids")["depmap_id"]
                           .agg(lambda s: sorted(set(s))))

# single model_id per profileid, only unambiguous ones (n==1)
profile_to_model_single = profile_to_model_set[profile_to_model_set.map(len) == 1].map(lambda s: s[0])

depmap_expr["model_id_set"] = depmap_expr["profileid"].map(profile_to_model_set)
depmap_expr["model_id"]     = depmap_expr["profileid"].map(profile_to_model_single)
depmap_expr["model_id_ambiguous"] = depmap_expr["model_id_set"].map(lambda s: isinstance(s, list) and len(s) > 1)

print("rows:", len(depmap_expr))
print("matched model_id (unambiguous):", depmap_expr["model_id"].notna().sum())
print("ambiguous (>1 model_id):       ", depmap_expr["model_id_ambiguous"].sum())
print("unmatched:                     ", depmap_expr["model_id_set"].isna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 16: Identity harmonisation — cardinality check: hpa_desc cellosaurus_id <-> cell_line
# ---------------------------------------------------------------------------
"""
Checks the cardinality of cellosaurus_id <-> cell_line in hpa_desc. Neither
column is normalised here — cell_line is compared as-is per instruction,
and cellosaurus_id is expected to already be in bare 'cvcl_xxxx' form from
the earlier normalisation pass.
"""

id_to_name = hpa_desc.dropna(subset=["cellosaurus id"]).groupby("cellosaurus id")["cell line"].nunique()
name_to_id = hpa_desc.dropna(subset=["cell line"]).groupby("cell line")["cellosaurus id"].nunique()

print("=== cellosaurus_id -> cell_line relationship ===")
print("cellosaurus_id with >1 cell_line:", (id_to_name > 1).sum(), "/", len(id_to_name))
print("max cell_line per cellosaurus_id:", id_to_name.max())
print()
print("=== cell_line -> cellosaurus_id relationship ===")
print("cell_line with >1 cellosaurus_id:", (name_to_id > 1).sum(), "/", len(name_to_id))
print("max cellosaurus_id per cell_line:", name_to_id.max())
print()
is_one_to_one = (id_to_name.max() == 1) and (name_to_id.max() == 1)
print("strictly 1:1 relationship:", is_one_to_one)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 17: Coverage auditing — hpa_desc vs hpa_rna cell_line overlap
# ---------------------------------------------------------------------------
"""
Checks overlap of cell_line values between hpa_desc and hpa_rna, exactly
as-is with no normalisation on either side (raw string match).
"""

desc_names = set(hpa_desc["cell line"].dropna())
rna_names  = set(hpa_rna["cell line"].dropna())

common     = desc_names & rna_names
only_desc  = desc_names - rna_names
only_rna   = rna_names - desc_names

print("distinct cell_line in hpa_desc:", len(desc_names))
print("distinct cell_line in hpa_rna: ", len(rna_names))
print("common:    ", len(common))
print("only desc: ", len(only_desc))
print("only rna:  ", len(only_rna))

In [ ]:
# ---------------------------------------------------------------------------
# Stage 18: Feature matrix assembly — attach model_id to hpa_desc via cellosaurus_id
# ---------------------------------------------------------------------------
"""
Adds model_id to hpa_desc by mapping its cellosaurus_id through the
rrid_by_ach roster. Since rrid_by_ach's rrids column is a set (one ACH can
have >1 RRID), it's exploded to a flat cvcl -> model_id lookup first.
Ambiguous CVCLs (mapping to >1 ACH, per the earlier cardinality check) are
preserved as a model_id_set rather than silently resolved via
drop_duplicates.
"""

exploded_cvcl = (rrid_by_ach[["depmap_id", "rrids"]]
                    .explode("rrids")
                    .dropna(subset=["rrids"]))

# full set of model_ids per cvcl (preserves ambiguity)
cvcl_to_model_set = (exploded_cvcl.groupby("rrids")["depmap_id"]
                        .agg(lambda s: sorted(set(s))))

# single model_id per cvcl, only unambiguous ones (n==1)
cvcl_to_model_single = cvcl_to_model_set[cvcl_to_model_set.map(len) == 1].map(lambda s: s[0])

hpa_desc["model_id"] = hpa_desc["cellosaurus id"].map(cvcl_to_model_set)

print("rows:", len(hpa_desc))
print("matched model_id (unambiguous):", hpa_desc["model_id"].notna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 19: Feature matrix assembly — attach model_id to hpa_rna via cell_line
# ---------------------------------------------------------------------------
"""
Maps model_id from hpa_desc into hpa_rna using the raw cell_line string,
no normalisation. hpa_desc's cellosaurus_id already carries a model_id
(from the earlier CVCL-based mapping); this attaches that same model_id
to hpa_rna by joining on the untouched cell_line name instead.
"""

# lookup: cell_line (as-is) -> model_id, from hpa_desc
cellline_to_model = (hpa_desc.dropna(subset=["model_id"])
                        .set_index("cell line")["model_id"])

hpa_rna["model_id"] = hpa_rna["cell line"].map(cellline_to_model)

print("rows:", len(hpa_rna))
print("matched model_id:", hpa_rna["model_id"].notna().sum())
print("unmatched:", hpa_rna["model_id"].isna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 20: Coverage auditing — proteomics depmap_id vs rrid_by_ach overlap
# ---------------------------------------------------------------------------
"""
Checks how many depmap_id values in proteomics overlap with the ACH roster
in rrid_by_ach.
"""

roster_achs      = set(rrid_by_ach["depmap_id"].dropna())
proteomics_achs  = set(proteomics["depmap_id"].dropna())

common      = proteomics_achs & roster_achs
only_prot   = proteomics_achs - roster_achs
only_roster = roster_achs - proteomics_achs

print("distinct depmap_id in proteomics:", len(proteomics_achs))
print("distinct depmap_id in roster:    ", len(roster_achs))
print("overlap (common):                ", len(common))

In [ ]:
# ---------------------------------------------------------------------------
# Stage 21: Coverage auditing — metabolomics depmap_id vs roster overlap
# ---------------------------------------------------------------------------
"""
Checks how many depmap_id values in metabolomics overlap with the ACH
roster in cell_line_connection.
"""

roster_achs      = set(rrid_by_ach["depmap_id"].dropna())
metabolomics_achs = set(metabolomics["depmap_id"].dropna())

common      = metabolomics_achs & roster_achs
only_metab  = metabolomics_achs - roster_achs
only_roster = roster_achs - metabolomics_achs

print("distinct depmap_id in metabolomics:", len(metabolomics_achs))
print("distinct depmap_id in roster:      ", len(roster_achs))
print("overlap (common):                  ", len(common))

In [ ]:
# ---------------------------------------------------------------------------
# Stage 23: Identity harmonisation — cardinality check: metabolomics ccle_id <-> depmap_id
# ---------------------------------------------------------------------------
"""
Checks the cardinality of ccle_id <-> depmap_id in metabolomics before
using it as a lookup for mirna. Confirms whether the ccle_id -> depmap_id
join used above is safe as a plain map, or whether some ccle_id values
resolve to more than one depmap_id.
"""

ccle_to_depmap = metabolomics.dropna(subset=["ccle_id"]).groupby("ccle_id")["depmap_id"].nunique()
depmap_to_ccle = metabolomics.dropna(subset=["depmap_id"]).groupby("depmap_id")["ccle_id"].nunique()

print("=== ccle_id -> depmap_id relationship ===")
print("ccle_id with >1 depmap_id:", (ccle_to_depmap > 1).sum(), "/", len(ccle_to_depmap))
print("max depmap_id per ccle_id:", ccle_to_depmap.max())
print()
print("=== depmap_id -> ccle_id relationship ===")
print("depmap_id with >1 ccle_id:", (depmap_to_ccle > 1).sum(), "/", len(depmap_to_ccle))
print("max ccle_id per depmap_id:", depmap_to_ccle.max())
print()
is_one_to_one = (ccle_to_depmap.max() == 1) and (depmap_to_ccle.max() == 1)
print("strictly 1:1 relationship:", is_one_to_one)

In [ ]:
# ---------------------------------------------------------------------------
# Stage 24: Coverage auditing — check mirna table orientation
# ---------------------------------------------------------------------------
"""
Checks the current shape and orientation of mirna — confirming whether
cell lines are rows and miRNAs are columns, or the reverse, and whether
the CCLE {name}_{tissue} identifier sits in the index or the columns.
"""

print("shape:", mirna.shape)
print("index name:  ", mirna.index.name)
print("index sample:", mirna.index[:5].tolist())
print("columns sample:", mirna.columns[:10].tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 25: Feature matrix assembly — reshape mirna to sample-rows/feature-columns
# ---------------------------------------------------------------------------
"""
Transposes mirna from miRNA-rows/CCLE-name-columns into the target wide
form: one row per CCLE cell line, one column per miRNA. 'name' (the miRNA
identifier) is set as the index before transposing so it becomes the
column headers post-transpose. 'description' is metadata, not a sample —
it's dropped before the transpose so it doesn't get treated as a row.
"""

mirna_wide = mirna.drop(columns=["description"]).set_index("name").T
mirna_wide.index.name = "ccle_name"
mirna_wide.columns.name = None
mirna_wide = mirna_wide.reset_index()
mirna = mirna_wide
mirna_wide_vals = mirna.set_index("ccle_name").apply(pd.to_numeric, errors="coerce")
print("non-numeric cells introduced:", mirna_wide_vals.isna().sum().sum())

print(mirna.shape)
mirna.iloc[:5, :5]

In [ ]:
# ---------------------------------------------------------------------------
# Stage 26: Feature matrix assembly — attach depmap_id to mirna via metabolomics ccle_id
# ---------------------------------------------------------------------------
"""
Adds depmap_id to mirna by matching its ccle_name against
metabolomics' ccle_id, then pulling the corresponding depmap_id across.
No normalisation applied — exact string match on the two CCLE identifiers
as they currently stand.
"""

# lookup: ccle_id -> depmap_id, from metabolomics
ccle_to_model = (metabolomics.dropna(subset=["depmap_id"])
                    .drop_duplicates("ccle_id")
                    .set_index("ccle_id")["depmap_id"])

mirna["depmap_id"] = mirna["ccle_name"].map(ccle_to_model)

print("rows:", len(mirna))
print("matched depmap_id:", mirna["depmap_id"].notna().sum())
print("unmatched:", mirna["depmap_id"].isna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 26: Identity harmonisation — attach ccle_name to rrid_by_ach via mirna
# ---------------------------------------------------------------------------
"""
Adds ccle_name to rrid_by_ach by mapping depmap_id back through
mirna's depmap_id -> ccle_name attachment (built in the previous
step). Since a depmap_id could in principle map to more than one
ccle_name, the set is preserved rather than assuming 1:1, consistent with
how rrids/profile_ids/geo_accessions were handled.
"""

ccle_by_model = (mirna.dropna(subset=["depmap_id"])
                    .groupby("depmap_id")["ccle_name"]
                    .agg(lambda s: sorted(set(s.dropna())))
                    .rename("ccle_names"))

rrid_by_ach = rrid_by_ach.merge(
    ccle_by_model, on="depmap_id", how="left"
)

print("rows:", len(rrid_by_ach))
print("matched ccle_name:", rrid_by_ach["ccle_names"].notna().sum())
print("unmatched:", rrid_by_ach["ccle_names"].isna().sum())
rrid_by_ach.head()

In [ ]:
# ---------------------------------------------------------------------------
# Stage 27: Identity harmonisation — rename roster to cell_line_connection
# ---------------------------------------------------------------------------
"""
Renames rrid_by_ach to cell_line_connection to better reflect its role as
the central identity bridge table (ACH <-> RRID/CVCL <-> profile_id <->
geo_accession) used throughout the rest of the pipeline.
"""

cell_line_connection = rrid_by_ach

# Adding the model_id to all the Tables

In [ ]:
# ---------------------------------------------------------------------------
# Stage 30: Feature matrix assembly — attach model_id to cellosaurus
# ---------------------------------------------------------------------------
"""
Adds model_id to cellosaurus by mapping its cellosaurus_accession through
the cell_line_connection roster's rrids set. Ambiguous CVCLs (mapping to
>1 ACH, per the earlier cardinality check) are preserved as model_id_set
rather than silently resolved.
"""

exploded_cvcl = (cell_line_connection[["depmap_id", "rrids"]]
                    .explode("rrids")
                    .dropna(subset=["rrids"]))

cvcl_to_model_set = (exploded_cvcl.groupby("rrids")["depmap_id"]
                        .agg(lambda s: sorted(set(s))))
cvcl_to_model_single = cvcl_to_model_set[cvcl_to_model_set.map(len) == 1].map(lambda s: s[0])

cellosaurus["model_id"] = cellosaurus["cellosaurus_accession"].map(cvcl_to_model_set)

print("rows:", len(cellosaurus))
print("matched model_id (unambiguous):", cellosaurus["model_id"].notna().sum())
print("unmatched:                     ", cellosaurus["model_id"].isna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 31: Identity harmonisation — rename depmap_profiles modelid -> model_id
# ---------------------------------------------------------------------------
"""
Renames depmap_profiles' modelid column to model_id, aligning it with the
model_id naming convention used across cell_line_connection and the other
modality tables.
"""

depmap_profiles = depmap_profiles.rename(columns={"modelid": "model_id"})

print(depmap_profiles.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 32: Identity harmonisation — split hpa_desc patient into gender/age
# ---------------------------------------------------------------------------
"""
Splits hpa_desc's 'patient' column into separate gender and age columns.
The raw format is inconsistent — some rows are 'gender, age' (e.g.
'male, 72'), others are age only (e.g. '13') with no gender recorded.
Both patterns are handled explicitly rather than assuming one fixed shape;
rows matching neither are flagged via NaN rather than silently dropped.
"""

patient_str = hpa_desc["patient"].astype("string").str.strip()

# pattern 1: "gender, age"  e.g. "male, 72"
gendered = patient_str.str.extract(r"^(?P<gender>\w+)\s*,\s*(?P<age>\d+)$")

# pattern 2: age only  e.g. "13"
age_only = patient_str.str.extract(r"^(?P<age>\d+)$")

hpa_desc["gender"] = gendered["gender"]
hpa_desc["age"] = pd.to_numeric(gendered["age"].combine_first(age_only["age"]), errors="coerce")

print("rows:", len(hpa_desc))
print("gender parsed:", hpa_desc["gender"].notna().sum())
print("age parsed:   ", hpa_desc["age"].notna().sum())

# rows with a patient value that matched neither pattern
had_value = patient_str.notna() & (patient_str != "")
failed = hpa_desc.loc[had_value & hpa_desc["age"].isna(), ["patient"]]
print("failed to parse:", len(failed))
print(failed.head(10))

In [ ]:
# ---------------------------------------------------------------------------
# Stage 33: Feature matrix assembly — attach depmap_id to mutations
# ---------------------------------------------------------------------------
"""
Adds depmap_id to mutations by mapping its profileid to its ACH via
cell_line_connection's profile_ids set. Since profile -> model_id was
confirmed 1:1 earlier, this is a safe direct lookup; the set-based pattern
is kept anyway for consistency with the other attachments and to catch
any future violation of that 1:1 assumption.
"""

exploded_profile = (cell_line_connection[["depmap_id", "profile_ids"]]
                       .explode("profile_ids")
                       .dropna(subset=["profile_ids"]))

profile_to_model_set = (exploded_profile.groupby("profile_ids")["depmap_id"]
                           .agg(lambda s: sorted(set(s))))
profile_to_model_single = profile_to_model_set[profile_to_model_set.map(len) == 1].map(lambda s: s[0])

mutations["depmap_id"] = mutations["profileid"].map(profile_to_model_set)

print("rows:", len(mutations))
print("matched depmap_id (unambiguous):", mutations["depmap_id"].notna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 34: Identity harmonisation — rename fusions modelid -> model_id
# ---------------------------------------------------------------------------
"""
Renames fusions' modelid column to model_id, aligning it with the
model_id naming convention used across cell_line_connection and the other
modality tables.
"""

fusions = fusions.rename(columns={"modelid": "model_id"})

print(fusions.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 35: Feature matrix assembly — attach model_id to geo_info via cell line name
# ---------------------------------------------------------------------------
"""
Adds model_id to geo_info by mapping its cell line name against
sample_info's stripped_cell_line_name -> depmap_id relationship (the
authoritative ACH source). Ambiguous names (mapping to >1 ACH) are
preserved as a model_id_set rather than silently resolved via
drop_duplicates.
"""

name_to_model_set = (sample_info.dropna(subset=["depmap_id"])
                        .groupby("stripped_cell_line_name")["depmap_id"]
                        .agg(lambda s: sorted(set(s.dropna()))))
name_to_model_single = name_to_model_set[name_to_model_set.map(len) == 1].map(lambda s: s[0])

geo_info["model_id"] = geo_info["cellline"].map(name_to_model_set)
geo_info["model_id_ambiguous"] = geo_info["model_id"].map(lambda s: isinstance(s, list) and len(s) > 1)

print("rows:", len(geo_info))
print("matched model_id (unambiguous):", geo_info["model_id"].notna().sum())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 36: Identity harmonisation — rename metabolomics depmap_id -> model_id
# ---------------------------------------------------------------------------
"""
Renames metabolomics' depmap_id column to model_id, aligning it with the
model_id naming convention used across cell_line_connection and the other
modality tables.
"""

metabolomics = metabolomics.rename(columns={"depmap_id": "model_id"})

print(metabolomics.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 37: Identity harmonisation — rename proteomics depmap_id -> model_id
# ---------------------------------------------------------------------------
"""
Renames proteomics' depmap_id column to model_id, aligning it with the
model_id naming convention used across cell_line_connection and the other
modality tables.
"""

proteomics = proteomics.rename(columns={"depmap_id": "model_id"})

print(proteomics.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 38: Identity harmonisation — rename mirna depmap_id -> model_id
# ---------------------------------------------------------------------------
"""
Renames mirna's depmap_id column to model_id, aligning it with the
model_id naming convention used across cell_line_connection and the other
modality tables.
"""

mirna = mirna.rename(columns={"depmap_id": "model_id"})
mirna = mirna
print(mirna.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 39: Identity harmonisation — rename signatures modelid -> model_id
# ---------------------------------------------------------------------------
"""
Renames signatures' modelid column to model_id, aligning it with the
model_id naming convention used across cell_line_connection and the other
modality tables.
"""

signatures = signatures.rename(columns={"modelid": "model_id"})

print(signatures.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 40 : Identity harmonisation — rename sample_info depmap_id -> model_id
# ---------------------------------------------------------------------------
"""
Renames sample_info's depmap_id column to model_id, aligning it with the
model_id naming convention used across the rest of the pipeline.
"""

sample_info = sample_info.rename(columns={"depmap_id": "model_id"})

print(sample_info.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 41: Identity harmonisation — rename mutations depmap_id -> model_id
# ---------------------------------------------------------------------------
"""
Renames mutations' depmap_id (and depmap_id_set) columns to model_id /
model_id_set, aligning them with the model_id naming convention used
across the rest of the pipeline.
"""

mutations = mutations.rename(columns={
    "depmap_id": "model_id"})

print(mutations.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 42: Identity harmonisation — rename cell_line_connection depmap_id -> model_id
# ---------------------------------------------------------------------------
"""
Renames cell_line_connection's depmap_id column to model_id, completing
the naming alignment across the entire pipeline. All prior joins that
referenced cell_line_connection["depmap_id"] will need model_id going
forward if re-run from a fresh kernel.
"""

cell_line_connection = cell_line_connection.rename(columns={"depmap_id": "model_id"})

print(cell_line_connection.columns.tolist())

In [ ]:
# ---------------------------------------------------------------------------
# Stage 43: Coverage auditing — check which tables lack a model_id column
# ---------------------------------------------------------------------------
"""
Scans all working tables for a model_id column, now that the renaming
pass (modelid/depmap_id -> model_id) has been applied across the board.
Flags any table still missing it, so it's clear what still needs attaching
before the AnnData/MuData assembly stage.
"""

all_tables = {
    "hpa_rna":               hpa_rna,
    "depmap_expr":           depmap_expr,
    "geo_expr":               geo_expr,
    "proteomics":            proteomics,
    "protein_map":           protein_map,
    "fusions":               fusions,
    "mutations":             mutations,
    "cellosaurus":           cellosaurus,
    "depmap_profiles":       depmap_profiles,
    "sample_info":           sample_info,
    "geo_info":              geo_info,
    "hpa_desc":              hpa_desc,
    "metabolomics":          metabolomics,
    "mirna":                 mirna,
    "signatures":            signatures
}

for name, df in all_tables.items():
    has_it = "model_id" in df.columns
    status = "OK" if has_it else "MISSING"
    print(f"{name:22s} -> model_id: {status}")

# Ach to Gene

In [ ]:
# ---------------------------------------------------------------------------
# Stage 44: Coverage auditing — ENSG column overlap between geo_expr and depmap_expr
# ---------------------------------------------------------------------------
"""
Checks how many ENSG-headed columns exist in geo_expr vs depmap_expr, and
how much overlap there is between the two gene sets — the gene axis
equivalent of the model_id coverage checks done earlier.
"""

geo_ensg    = {c for c in geo_expr.columns if c.startswith("ensg")}
depmap_ensg = {c for c in depmap_expr.columns if c.startswith("ensg")}

common     = geo_ensg & depmap_ensg
only_geo   = geo_ensg - depmap_ensg
only_depmap = depmap_ensg - geo_ensg

print("ensg columns in geo_expr:    ", len(geo_ensg))
print("ensg columns in depmap_expr: ", len(depmap_ensg))
print("common to both:              ", len(common))
print("only in geo_expr:            ", len(only_geo))
print("only in depmap_expr:         ", len(only_depmap))
print(f"geo coverage of depmap genes:    {len(common)/len(depmap_ensg)*100:.1f}%")
print(f"depmap coverage of geo genes:    {len(common)/len(geo_ensg)*100:.1f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Stage 3: Coverage auditing — full modality coverage per cell line (model_id)
# ---------------------------------------------------------------------------
"""
Builds a per-model_id coverage matrix across the modalities that make up
one "complete cycle" for a cell line:
    - gene_expression : present in depmap_expr and/or geo_expr
    - mutation        : present in mutations
    - fusion          : present in fusions
    - protein         : present in proteomics
    - hpa_rna         : present in hpa_rna
    - metabolomics    : present in metabolomics
    - mirna           : present in mirna
    - signatures      : present in signatures

model_id columns may hold list values (from the earlier model_id_set
attach pattern) rather than plain scalars. Ambiguous rows (list length > 1)
are NOT dropped here — each ACH in the ambiguous list is credited with
having that modality's data, since the underlying data genuinely belongs
to whichever of those ACHs it is (we just don't yet know which one), so
excluding both would understate coverage for lines caught in an identity
ambiguity.
"""

def clean_id_set(series):
    """Flatten scalar and list values into one set of ids. Ambiguous
    (length>1) lists contribute every id they contain, rather than being
    dropped, since the modality's data genuinely belongs to both of them
    (the ambiguity is unresolved, so both candidate ACHs are credited)."""
    s = series.dropna()
    ids = set()
    for x in s:
        if isinstance(x, list):
            ids.update(x)
        else:
            ids.add(x)
    return ids

modality_ids = {
    "gene_expression_depmap": clean_id_set(depmap_expr["model_id"]),
    "gene_expression_geo":    clean_id_set(geo_expr["model_id"]),
    "mutation":               clean_id_set(mutations["model_id"]),
    "fusion":                 clean_id_set(fusions["model_id"]),
    "protein":                clean_id_set(proteomics["model_id"]),
    "hpa_rna":                clean_id_set(hpa_rna["model_id"]),
    "metabolomics":           clean_id_set(metabolomics["model_id"]),
    "mirna":                  clean_id_set(mirna["model_id"]),
    "signatures":             clean_id_set(signatures["model_id"]),
}

# gene_expression counts as present if EITHER depmap or geo has it
gene_expr_ids = modality_ids["gene_expression_depmap"] | modality_ids["gene_expression_geo"]

coverage_sets = {
    "gene_expression": gene_expr_ids,
    "mutation":        modality_ids["mutation"],
    "fusion":          modality_ids["fusion"],
    "protein":         modality_ids["protein"],
    "hpa_rna":         modality_ids["hpa_rna"],
    "metabolomics":    modality_ids["metabolomics"],
    "mirna":           modality_ids["mirna"],
    "signatures":      modality_ids["signatures"]
}

all_ids = clean_id_set(cell_line_connection["model_id"])

print(f"total model_id in roster: {len(all_ids)}\n")
for name, ids in coverage_sets.items():
    overlap = ids & all_ids
    print(f"{name:18s} -> {len(overlap):5d} / {len(all_ids)} model_ids "
          f"({len(overlap)/len(all_ids)*100:5.1f}%)")

# build the coverage matrix: one row per model_id, one bool col per modality
coverage_matrix = pd.DataFrame({"model_id": sorted(all_ids)})
for name, ids in coverage_sets.items():
    coverage_matrix[name] = coverage_matrix["model_id"].isin(ids)

coverage_matrix["n_modalities"] = coverage_matrix[list(coverage_sets.keys())].sum(axis=1)
coverage_matrix["complete_cycle"] = coverage_matrix["n_modalities"] == len(coverage_sets)

print("\nmodel_ids with ALL 8 modalities (complete cycle):", coverage_matrix["complete_cycle"].sum())
print("\ndistribution of n_modalities present:")
print(coverage_matrix["n_modalities"].value_counts().sort_index(ascending=False))

coverage_matrix

# Missing Data % Audit

In [ ]:
# ---------------------------------------------------------------------------
# Stage 4: Coverage auditing — missing % summary across all tables (tabular)
# ---------------------------------------------------------------------------
"""
Computes missing-value percentage per column for every table in the
pipeline as a single wide summary table: one row per (table, column),
sorted by missing_pct descending so the worst gaps surface first.
"""

all_tables = {
    "hpa_rna":               hpa_rna,
    "depmap_expr":           depmap_expr,
    "geo_expr":              geo_expr,
    "proteomics":            proteomics,
    "protein_map":           protein_map,
    "fusions":               fusions,
    "mutations":             mutations,
    "cellosaurus":           cellosaurus,
    "depmap_profiles":       depmap_profiles,
    "sample_info":           sample_info,
    "geo_info":              geo_info,
    "hpa_desc":              hpa_desc,
    "metabolomics":          metabolomics,
    "mirna":                 mirna,
    "signatures":            signatures
}

rows = []
for name, df in all_tables.items():
    for col in df.columns:
        rows.append({
            "table": name,
            "column": col,
            "n_rows": len(df),
            "n_missing": df[col].isna().sum(),
            "missing_pct": round(df[col].isna().mean() * 100, 1),
        })

missing_summary = pd.DataFrame(rows).sort_values("missing_pct", ascending=False).reset_index(drop=True)
print(missing_summary.shape)
missing_summary

In [ ]:
# ---------------------------------------------------------------------------
# Stage 45: Coverage auditing — drop 100%-missing columns from all tables
# ---------------------------------------------------------------------------
"""
Drops any column that is 100% missing in its respective table, using the
missing_summary computed above. Applied per-table (a column name might be
100% missing in one table but fine in another), and audited so exactly
what got dropped is visible rather than silently disappearing.
"""

fully_missing = missing_summary[missing_summary["missing_pct"] == 100.0]

for name, df in all_tables.items():
    cols_to_drop = fully_missing.loc[fully_missing["table"] == name, "column"].tolist()
    if cols_to_drop:
        all_tables[name] = df.drop(columns=cols_to_drop)
        print(f"{name}: dropped {len(cols_to_drop)} fully-missing cols -> {cols_to_drop}")

# reassign back to the individual variables since all_tables held references
hpa_rna               = all_tables["hpa_rna"]
depmap_expr           = all_tables["depmap_expr"]
geo_expr              = all_tables["geo_expr"]
proteomics            = all_tables["proteomics"]
protein_map           = all_tables["protein_map"]
fusions               = all_tables["fusions"]
mutations             = all_tables["mutations"]
cellosaurus           = all_tables["cellosaurus"]
depmap_profiles       = all_tables["depmap_profiles"]
sample_info           = all_tables["sample_info"]
geo_info              = all_tables["geo_info"]
hpa_desc              = all_tables["hpa_desc"]
metabolomics          = all_tables["metabolomics"]
mirna                 = all_tables["mirna"]
signatures            = all_tables["signatures"]

print("\ntotal columns dropped across all tables:", len(fully_missing))

In [ ]:
# ---------------------------------------------------------------------------
# Stage 46: Coverage auditing — check numeric scale per column per table
# ---------------------------------------------------------------------------
"""
Computes basic scale statistics (min, max, mean, median, std) for every
numeric column across all tables — useful for spotting which columns are
raw counts, log-ratios, z-scored, percentages, etc. before deciding on a
per-modality normalization strategy. Non-numeric columns are skipped.
"""

rows = []
for name, df in all_tables.items():
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        rows.append({
            "table": name,
            "column": col,
            "n_non_null": len(s),
            "min": s.min(),
            "max": s.max(),
            "mean": round(s.mean(), 3),
            "median": round(s.median(), 3),
            "std": round(s.std(), 3),
        })

scale_summary = pd.DataFrame(rows)
scale_summary